*0.3 Classical NLP*

# BM25

**The situation.** TF-IDF search is live. Two complaints: an article that mentions "refund" twenty times ranks far above one that mentions it three times, though both are about refunds; and long articles are still favoured for some queries. Elasticsearch and every search engine fixed both years ago with a different formula.

**BM25.** A refinement of TF-IDF for ranking. Term frequency *saturates*: the 20th mention adds almost nothing over the 5th. Document length is normalised against the average length in the corpus. Two parameters — `k1` for saturation, `b` for length — with defaults that work almost everywhere. It is the default ranking in Elasticsearch, OpenSearch and Lucene, and the "keyword half" of hybrid RAG.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
from rank_bm25 import BM25Okapi

articles = [
    (
        "refund refund refund refund refund refund refund refund refund refund refund refund "
        "refund refund refund refund refund refund refund refund"
    ),
    "How to request a refund for a duplicate charge, and when the refund arrives",
    "Why was I charged twice for the same order",
    "Change the email address on your account",
    "Reset your password from the login page",
    "Enable two-factor authentication for your team",
    "Export your data as a CSV file",
    "Invite a colleague to your workspace",
]
labels = [
    "'refund' × 20",
    "refund article",
    "charged twice article",
    "email article",
    "password",
    "two-factor",
    "export",
    "invite",
]


def tokens(text: str) -> list[str]:
    result = []
    for word in text.lower().split():
        result.append(word.strip(",.?"))
    return result


corpus_tokens = []
for article in articles:
    corpus_tokens.append(tokens(article))
bm25 = BM25Okapi(corpus_tokens)

scores = bm25.get_scores(tokens("refund duplicate charge"))
for score, label in sorted(zip(scores, labels), reverse=True):
    print(f"{score:.2f}  {label}")
assert labels[int(scores.argmax())] == "refund article"

3.84  refund article
2.10  'refund' × 20
0.00  two-factor
0.00  password
0.00  invite
0.00  export
0.00  email article
0.00  charged twice article


**Reading the output.** The real refund article wins. The document that only repeats "refund" twenty times does not — its twenty mentions saturate to roughly the value of two or three, and it matches none of the other query words.

```
                score
TF-IDF   ──▶    ╱           keeps growing with every repeat
BM25     ──▶   ╭───────     saturates: the 20th "refund" adds ~nothing
             count of the word in the document →
```

**The rule to remember.** BM25 is TF-IDF done right for ranking: saturating term frequency and length normalisation. Use it as the keyword side of any search.

| Use it when | Don't when | Instead use |
|---|---|---|
| keyword search, hybrid RAG, any "find documents containing these terms" | the query uses different words than the document | embeddings, or hybrid (BM25 + vectors, merged) |

**Watch out**
- `rank_bm25` is for prototypes and small corpora (it scores every document). For production use Elasticsearch/OpenSearch or a vector database with a BM25 option.
- Tokenization decides everything: stem (0.3 item 7) both the corpus and the query, the same way.
- Scores are not comparable across queries; do not threshold on them globally.